# 04 Extract Embeddings

**Purpose.** Pass images through frozen ImageNet encoders (ResNet-18, EfficientNet-B0, ViT-B/16); cache embeddings.

**Thesis reference.** §4.8 Frozen Encoder Pipeline

**Inputs / outputs.** See the data-layout table in the repository README. Paths near the top of the notebook (e.g. `PROFILE_DIR`, `IMAGE_ROOT`, `LABELS`) are relative to the repository root and assume the folder layout described there.

---

In [21]:
import torch
print("torch", torch.__version__)

torch 2.8.0+cu128


In [11]:
!pip install --no-deps --no-cache-dir torchvision==0.23.0 --index-url https://download.pytorch.org/whl/cu128

Looking in indexes: https://download.pytorch.org/whl/cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 39.6 MB/s eta 0:00:00a 0:00:01


In [13]:
import torch, torchvision
import torchvision.models as tvm
print("torch      ", torch.__version__)
print("torchvision", torchvision.__version__)
print("threads    ", torch.get_num_threads())
print()
for n in ["resnet18","resnet50","vgg16_bn","efficientnet_b0",
          "vit_b_16","swin_t","convnext_tiny"]:
    print(f"  {n:18s} {'yes' if hasattr(tvm,n) else 'no'}")

from pathlib import Path
root = Path("images224")
print(f"\n{root} exists: {root.is_dir()}")
if root.is_dir():
    for d in sorted(root.iterdir()):
        if d.is_dir():
            print(f"  {d.name:24s} {len(list(d.glob('*.png'))):5d} png")

import shutil
f = shutil.disk_usage("/").free/1e9
print(f"\nfree disk: {f:.1f} GB")

torch       2.8.0+cu128
torchvision 0.23.0+cu128
threads     32

  resnet18           yes
  resnet50           yes
  vgg16_bn           yes
  efficientnet_b0    yes
  vit_b_16           yes
  swin_t             yes
  convnext_tiny      yes

images224 exists: True
  .ipynb_checkpoints           0 png
  grayscale_heatmap          597 png
  lineplot_intensity         597 png
  pseudocolor_heatmap        597 png
  recurrence_plot            597 png
  rgb_mz_split               597 png
  scatterplot                597 png
  tic_plot                   597 png

free disk: 1.1 GB


In [22]:
IMAGE_ROOT      = "images224"     # written by 01_combined_pass
# LABELS          = "labels/labels_severity_nondiabetic.json"
LABELS           = "labels/labels_detect_any_ckd.json"
# OUT_DIR         = "embeddings"
OUT_DIR = "embeddings_detect"

REPRESENTATIONS = ["grayscale_heatmap", "rgb_mz_split"]

MODELS          = ["resnet18", "efficientnet_b0", "vit_b_16"]

BATCH_SIZE      = 8
THREADS         = 0        # 0 = let torch decide
WORKERS         = 0        # dataloader workers; keep 0 on a shared server

ARGS = ["--image-root", IMAGE_ROOT,
        "--labels", LABELS,
        "--out-dir", OUT_DIR,
        "--models", *MODELS,
        "--representations", *REPRESENTATIONS,
        "--batch-size", str(BATCH_SIZE),
        "--threads", str(THREADS),
        "--workers", str(WORKERS)]

In [23]:
import argparse
import json
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torchvision.models as tvm

# name -> (constructor, weights enum, how to strip the classifier head)
REGISTRY = {
    "resnet18":        (tvm.resnet18,        "ResNet18_Weights",        "fc"),
    "resnet50":        (tvm.resnet50,        "ResNet50_Weights",        "fc"),
    "vgg16_bn":        (tvm.vgg16_bn,        "VGG16_BN_Weights",        "classifier"),
    "efficientnet_b0": (tvm.efficientnet_b0, "EfficientNet_B0_Weights", "classifier"),
    "vit_b_16":        (tvm.vit_b_16,        "ViT_B_16_Weights",        "heads"),
    "swin_t":          (tvm.swin_t,          "Swin_T_Weights",          "head"),
    "convnext_tiny":   (tvm.convnext_tiny,   "ConvNeXt_Tiny_Weights",   "classifier"),
}

DEFAULT_MODELS = ["resnet18", "efficientnet_b0", "vit_b_16"]
DEFAULT_REPRESENTATIONS = ["grayscale_heatmap", "rgb_mz_split"]

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def available_models():
    ok = []
    for name, (ctor, wname, _) in REGISTRY.items():
        if hasattr(tvm, name) and hasattr(tvm, wname):
            ok.append(name)
    return ok

def build(model_name):
    """Return (model in eval mode with head removed, expected input size)."""
    ctor, wname, head = REGISTRY[model_name]
    weights = getattr(tvm, wname).DEFAULT
    model = ctor(weights=weights)

    # replace the classifier with identity so forward() returns features
    if head == "classifier" and isinstance(getattr(model, head), nn.Sequential):
        # vgg / efficientnet / convnext: keep everything but the final Linear
        seq = list(getattr(model, head))
        while seq and not isinstance(seq[-1], nn.Linear):
            seq.pop()
        if seq:
            seq.pop()
        setattr(model, head, nn.Sequential(*seq) if seq else nn.Identity())
    else:
        setattr(model, head, nn.Identity())

    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)

    size = 224
    try:
        size = weights.transforms().crop_size[0]
    except Exception:
        pass
    return model, size

class ImgSet(Dataset):
    def __init__(self, image_dir, pids, size):
        self.dir, self.pids = Path(image_dir), pids
        self.tf = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])

    def __len__(self):
        return len(self.pids)

    def __getitem__(self, i):
        im = Image.open(self.dir / f"{self.pids[i]}.png").convert("RGB")
        return self.tf(im)

@torch.no_grad()
def extract(model_name, image_dir, pids, batch_size=8, workers=0):
    model, size = build(model_name)
    dl = DataLoader(ImgSet(image_dir, pids, size), batch_size=batch_size,
                    shuffle=False, num_workers=workers)
    feats, t0, done = [], time.time(), 0
    for x in dl:
        out = model(x)
        if out.ndim > 2:                       # safety net for pooled maps
            out = out.flatten(1)
        feats.append(out.cpu().numpy())
        done += len(x)
        print(f"    {done}/{len(pids)}", end="\r", flush=True)
    return np.concatenate(feats, 0), time.time() - t0

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--image-root", required=True)
    ap.add_argument("--labels", required=True)
    ap.add_argument("--out-dir", default="embeddings")
    ap.add_argument("--models", nargs="+", default=DEFAULT_MODELS)
    ap.add_argument("--representations", nargs="+",
                    default=DEFAULT_REPRESENTATIONS)
    ap.add_argument("--batch-size", type=int, default=8)
    ap.add_argument("--threads", type=int, default=0)
    ap.add_argument("--workers", type=int, default=0)
    args = ap.parse_args()

    if args.threads > 0:
        torch.set_num_threads(args.threads)

    with open(args.labels) as fh:
        obj = json.load(fh)
    labels = obj.get("labels", obj)

    out = Path(args.out_dir)
    out.mkdir(parents=True, exist_ok=True)

    have = available_models()
    models = [m for m in args.models if m in have]
    missing = [m for m in args.models if m not in have]
    if missing:
        print(f"not available in this torchvision: {', '.join(missing)}")
    if not models:
        raise SystemExit(f"No usable models. Available here: {', '.join(have)}")

    print(f"torch {torch.__version__}, threads {torch.get_num_threads()}")
    print(f"models: {', '.join(models)}")
    print(f"representations: {', '.join(args.representations)}\n")

    for rep in args.representations:
        image_dir = Path(args.image_root) / rep
        if not image_dir.is_dir():
            print(f"[skip] {rep}: {image_dir} not found")
            continue
        pids = [p for p in sorted(labels)
                if (image_dir / f"{p}.png").exists()]
        if not pids:
            print(f"[skip] {rep}: no images matching the label file")
            continue
        print(f"{rep}: {len(pids)} patients")

        for mname in models:
            dest = out / f"{rep}__{mname}.npy"
            if dest.exists():
                print(f"  {mname}: already done, skipping")
                continue
            try:
                emb, secs = extract(mname, image_dir, pids,
                                    args.batch_size, args.workers)
            except Exception as e:
                print(f"  {mname}: FAILED ({type(e).__name__}: {e})")
                continue
            np.save(dest, emb)
            with open(dest.with_suffix(".json"), "w") as fh:
                json.dump({"representation": rep, "model": mname,
                           "pids": pids, "dim": int(emb.shape[1]),
                           "seconds": round(secs, 1)}, fh, indent=1)
            print(f"  {mname}: {emb.shape} in {secs/60:.1f} min -> {dest.name}")
        print()

    print(f"Embeddings written to {out}/")

In [24]:
import sys
sys.argv = ["02_extract_embeddings_tv.py"] + ARGS
main()

torch 2.8.0+cu128, threads 32
models: resnet18, efficientnet_b0, vit_b_16
representations: grayscale_heatmap, rgb_mz_split

grayscale_heatmap: 596 patients
  resnet18: (596, 512) in 0.2 min -> grayscale_heatmap__resnet18.npy
  efficientnet_b0: (596, 1280) in 0.1 min -> grayscale_heatmap__efficientnet_b0.npy
  vit_b_16: (596, 768) in 0.2 min -> grayscale_heatmap__vit_b_16.npy

rgb_mz_split: 596 patients
  resnet18: (596, 512) in 0.2 min -> rgb_mz_split__resnet18.npy
  efficientnet_b0: (596, 1280) in 0.1 min -> rgb_mz_split__efficientnet_b0.npy
  vit_b_16: (596, 768) in 0.2 min -> rgb_mz_split__vit_b_16.npy

Embeddings written to embeddings_detect/


In [25]:
# Fast --- runs on cached embeddings, no images touched.
import json, numpy as np
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold

lab = json.load(open(LABELS))["labels"]
ID_AUC, PROFILE_AUC = 0.519, 0.644     # patient-ID baseline; 0.02 Da profile

def cv_auc(X, y, n_repeats=5):
    mk = lambda: Pipeline([("s", StandardScaler()),
                           ("c", LogisticRegression(C=0.1, max_iter=3000,
                                                    class_weight="balanced"))])
    a = []
    for tr, te in RepeatedStratifiedKFold(n_splits=5, n_repeats=n_repeats,
                                          random_state=42).split(X, y):
        if len(np.unique(y[te])) < 2:
            continue
        a.append(roc_auc_score(y[te],
                 mk().fit(X[tr], y[tr]).predict_proba(X[te])[:, 1]))
    return float(np.mean(a))

print(f"{'representation':20s} {'model':18s} {'dim':>5} {'AUC':>7} "
      f"{'vs ID':>8} {'vs profile':>11}")
print("-" * 74)
rows = []
for f in sorted(Path(OUT_DIR).glob("*.npy")):
    meta = f.with_suffix(".json")
    if not meta.exists():
        continue
    info = json.load(open(meta))
    X = np.load(f)
    y = np.array([lab[p] for p in info["pids"]])
    if len(np.unique(y)) < 2:
        continue
    auc = cv_auc(X, y)
    print(f"{info['representation']:20s} {info['model']:18s} {X.shape[1]:5d} "
          f"{auc:7.4f} {auc-ID_AUC:+8.4f} {auc-PROFILE_AUC:+11.4f}")
    rows.append((info["representation"], info["model"], X.shape[1], auc))

if rows:
    b = max(rows, key=lambda r: r[3])
    print(f"\nBest image-based result: {b[0]} / {b[1]} = {b[3]:.4f}")
    print(f"0.02 Da profile reached {PROFILE_AUC:.4f} on the same patients.")
    print(f"The image is ~357x coarser in mass (7.14 Da/px vs 0.02 Da).\n")
    if b[3] >= PROFILE_AUC - 0.02:
        print("-> The image representation RETAINS the signal despite the")
        print("   compression. Deep features compensate for the resolution loss.")
    elif b[3] > ID_AUC + 0.05:
        print(f"-> Partial retention: {PROFILE_AUC-b[3]:.3f} AUC below the profile,")
        print(f"   but {b[3]-ID_AUC:.3f} above the patient-ID baseline. The image")
        print("   keeps some signal and loses some.")
    else:
        print("-> The image representation LOSES the signal. Compression to")
        print("   224 px costs information that deep features do not recover.")

representation       model                dim     AUC    vs ID  vs profile
--------------------------------------------------------------------------
grayscale_heatmap    efficientnet_b0     1280  0.6784  +0.1594     +0.0344
grayscale_heatmap    resnet18             512  0.6353  +0.1163     -0.0087
grayscale_heatmap    vit_b_16             768  0.7071  +0.1881     +0.0631
rgb_mz_split         efficientnet_b0     1280  0.6829  +0.1639     +0.0389
rgb_mz_split         resnet18             512  0.6028  +0.0838     -0.0412
rgb_mz_split         vit_b_16             768  0.6361  +0.1171     -0.0079

Best image-based result: grayscale_heatmap / vit_b_16 = 0.7071
0.02 Da profile reached 0.6440 on the same patients.
The image is ~357x coarser in mass (7.14 Da/px vs 0.02 Da).

-> The image representation RETAINS the signal despite the
   compression. Deep features compensate for the resolution loss.
